# MonIA FramePack Kaggle benchmark
Candidate-only I2V benchmark. No narrative authority, no auto-publish to live game.
Probe revision 2 — trigger isolated Kaggle capability run.\n

In [ ]:
!git clone -q --depth 1 https://github.com/lllyasviel/FramePack.git /kaggle/working/FramePack
%cd /kaggle/working/FramePack
%pip -q install -r requirements.txt
%pip -q install --no-deps xformers
print('MONIA_XFORMERS_INSTALLED_POST_REQUIREMENTS')
import subprocess, sys
_xf=subprocess.run([sys.executable,'-c',"import numpy, scipy, xformers; print('MONIA_XFORMERS_CLEAN_PROCESS_OK', numpy.__version__, scipy.__version__, xformers.__version__)"],text=True,capture_output=True)
print(_xf.stdout.strip())
assert _xf.returncode==0, _xf.stderr
print('FramePack installed')


In [ ]:
# This benchmark intentionally uses the official FramePack implementation rather than reimplementing its sampler.
import torch
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('VRAM GB:',round(torch.cuda.get_device_properties(0).total_memory/1024**3,2) if torch.cuda.is_available() else 0)
assert torch.cuda.is_available()
print('MONIA_FRAMEPACK_READY')


In [ ]:
import os, shutil, subprocess
os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'
os.environ['HF_HUB_DISABLE_XET']='1'
os.environ['HF_HUB_ENABLE_HF_TRANSFER']='0'
# Persistent-weight contract: the GPU worker must consume a Kaggle-mounted cache,
# never reconstruct the 20+ GB model inside /kaggle/working.
from pathlib import Path
_input=Path('/kaggle/input')
assert _input.exists(), 'Kaggle input mount is unavailable'
_candidates=[p for p in _input.rglob('*') if p.is_dir() and ('FramePack_F1_I2V_HY_20250503' in p.name or 'framepack-f1' in str(p).lower())]
print('MONIA_KAGGLE_INPUT_TOP', [str(p) for p in list(_input.iterdir())[:20]])
assert _candidates, 'Persistent FramePack dataset is mounted but model directory layout was not recognized'
_model_dir=next((p for p in _candidates if (p/'transformer').exists() or (p/'model_index.json').exists()), _candidates[0])
os.environ['MONIA_FRAMEPACK_MODEL_DIR']=str(_model_dir)
os.environ['TRANSFORMERS_OFFLINE']='1'
os.environ['HF_HUB_OFFLINE']='1'
print('MONIA_FRAMEPACK_PERSISTENT_WEIGHTS', _model_dir)
from pathlib import Path
demo=Path('/kaggle/working/FramePack/demo_gradio_f1.py')
s=demo.read_text()
# Tesla T4 (sm75) cannot execute FramePack/xFormers bf16 kernels; force fp16.
s=s.replace('torch.bfloat16','torch.float16')
print('MONIA_T4_FP16_PATCH enabled')
# Bind every heavyweight FramePack dependency to the persistent Kaggle dataset.
from pathlib import Path as _MoniaPath
_dataset_root=_MoniaPath('/kaggle/input/monia-framepack-f1-weights')
_hunyuan=_dataset_root/'HunyuanVideo'
_flux=_dataset_root/'flux_redux_bfl'
_transformer=_dataset_root/'FramePack_F1_I2V_HY_20250503'
for _p in (_hunyuan,_flux,_transformer):
    assert _p.exists(), f'Missing persistent FramePack dependency: {_p}'
s=s.replace('"hunyuanvideo-community/HunyuanVideo"', repr(str(_hunyuan)))
s=s.replace('"lllyasviel/flux_redux_bfl"', repr(str(_flux)))
s=s.replace("'lllyasviel/FramePack_F1_I2V_HY_20250503'", repr(str(_transformer)))
print('MONIA_FRAMEPACK_OFFLINE_BINDINGS_OK')
# Tesla T4 cannot run xFormers attention in bfloat16; cast attention tensors only to fp16.
_attn_file=Path('/kaggle/working/FramePack/diffusers_helper/models/hunyuan_video_packed.py')
_attn_src=_attn_file.read_text()
_needle='x = xformers_attn_func(q, k, v)'
assert _needle in _attn_src, 'FramePack xFormers attention call not found'
_replacement="_monia_attn_dtype = q.dtype\n        x = xformers_attn_func(q.to(torch.float16), k.to(torch.float16), v.to(torch.float16)).to(_monia_attn_dtype)"
_attn_src=_attn_src.replace(_needle,_replacement)
_attn_file.write_text(_attn_src)
print('MONIA_XFORMERS_T4_FP16_PATCH_OK')
# Tesla T4 (sm75) cannot run xFormers attention with BF16; cast only Q/K/V attention to FP16.
_attn_file=Path('/kaggle/working/FramePack/diffusers_helper/models/hunyuan_video_packed.py')
_attn_src=_attn_file.read_text()
_xf_old='x = xformers_attn_func(q, k, v)'
_xf_new="x = xformers_attn_func(q.to(torch.float16), k.to(torch.float16), v.to(torch.float16)).to(q.dtype)"
assert _xf_old in _attn_src, 'Expected FramePack xFormers attention call not found'
_attn_file.write_text(_attn_src.replace(_xf_old,_xf_new,1))
print('MONIA_XFORMERS_T4_FP16_PATCH')
needle='block.launch('
assert needle in s
inject=r'''# MONIA_HEADLESS_FRAMEPACK
import shutil, json
from pathlib import Path as _MoniaPath
from PIL import Image as _MoniaImage
_src='/kaggle/working/marion-framepack-source.png'
_anchor_candidates=[_MoniaPath('/kaggle/input/monia-framepack-f1-weights/marion-framepack-anchor.png'),_MoniaPath('/kaggle/input/monia-framepack-f1-weights/references/marion-framepack-anchor.png')]
_anchor=next((p for p in _anchor_candidates if p.exists()),None)
if _anchor is not None:
    shutil.copy2(_anchor,_src)
else:
    import urllib.request as _urlreq, base64 as _b64
    _bootstrap='https://raw.githubusercontent.com/vartcom38-collab/marion-lucas-game/main/private-reference-bootstrap/marion/marion-framepack-probe.jpg'
    _payload=_urlreq.urlopen(_bootstrap,timeout=30).read()
    _MoniaPath(_src).write_bytes(_payload)
    print('MONIA_MARION_ANCHOR_BOOTSTRAPPED_FROM_REPO',len(_payload))
assert _MoniaPath(_src).exists() and _MoniaPath(_src).stat().st_size>4000, 'Durable Marion anchor could not be materialized'
_img=np.array(_MoniaImage.open(_src).convert('RGB'))
_prompt='Photorealistic cinematic micro-shot. Marion is seated in her warm apartment holding a white mug. One principal action only: she hears a quiet sound near the balcony and naturally turns her head toward it. Subtle breathing and one natural blink; restrained realistic motion. External invisible cinematic camera, stable medium-close framing. She never touches, holds, addresses, or looks into the camera. Preserve face, hair, cream knit sweater, apartment and morning lighting from the reference. No dialogue. End settled after the head turn, ready for a clean editorial cut.'
_last=None
for _event in process(_img,_prompt,'',221101,1.0,9,25,1.0,10.0,0.0,12.0,False,1):
    if isinstance(_event,tuple) and _event and isinstance(_event[0],str) and _event[0].endswith('.mp4'): _last=_event[0]
assert _last and os.path.isfile(_last), 'FramePack produced no MP4'
_out=_MoniaPath('/kaggle/working/monia-framepack-output'); _out.mkdir(exist_ok=True)
shutil.copy2(_last,_out/'shot-01.mp4'); shutil.copy2(_src,_out/'source.png')
(_out/'result.json').write_text(json.dumps({'jobId':'marion-framepack-f1-001','state':'candidate','candidateOnly':True,'narrativeAuthority':False,'router':'framepack-f1','clips':['shot-01.mp4'],'humanApprovalRequired':True},indent=2))
print('MONIA_FRAMEPACK_VIDEO_READY',_last)
raise SystemExit(0)
'''
demo.write_text(s.replace(needle,inject+'\n'+needle,1))
_run=subprocess.run(['python','demo_gradio_f1.py'],cwd='/kaggle/working/FramePack',text=True)
assert _run.returncode==0, f'FramePack subprocess failed with code {_run.returncode}'
_verified=Path('/kaggle/working/monia-framepack-output/shot-01.mp4')
assert _verified.exists(), 'FramePack subprocess exited without candidate MP4'
_probe=subprocess.run(['ffprobe','-v','error','-show_entries','format=duration,size:stream=codec_name,width,height,nb_frames','-of','json',str(_verified)],text=True,capture_output=True)
assert _probe.returncode==0, f'ffprobe failed: {_probe.stderr}'
_media=json.loads(_probe.stdout)
(_out/'ffprobe.json').write_text(json.dumps(_media,indent=2))
_streams=_media.get('streams') or []
assert _streams, 'FramePack MP4 contains no video stream'
_fmt=_media.get('format') or {}
_duration=float(_fmt.get('duration') or 0)
_size=int(_fmt.get('size') or _verified.stat().st_size)
assert _duration>0.5, f'FramePack MP4 duration is invalid: {_duration}'
print('MONIA_FRAMEPACK_MEDIA_PROBE', json.dumps({'duration':_duration,'size':_size,'stream':_streams[0]}))
for _label,_ss in [('first','0.0'),('middle',str(max(0.0,_duration/2))),('last',str(max(0.0,_duration-0.05)))]:
    _png=_out/f'{_label}.png'
    _ff=subprocess.run(['ffmpeg','-y','-ss',_ss,'-i',str(_verified),'-frames:v','1',str(_png)],text=True,capture_output=True)
    print('MONIA_FRAMEPACK_FRAME',_label,_png,_png.stat().st_size if _png.exists() else 0)
print('MONIA_FRAMEPACK_CANDIDATE_MEDIA_READY',_verified,_verified.stat().st_size)
